# Pre-Lab 5: Forced vibration
## Submit one document — not just a completed run
Upload **one Word/PDF to Canvas before your own Lab 5 section**. Put your name and section at the top. Use headings **A–E**. Include your calculations, short explanations and these **three figures**:

| Figure | File to download | Required contents |
|---|---|---|
| 1, part B | `PreLab5_Impulse_Comparison.png` | J = 1 and J = 2 N s responses on the same axes |
| 2, part C | `PreLab5_Transient_Steady.png` | Full response, steady response and their transient difference |
| 3, part D | `PreLab5_Frequency_Response.png` | Predicted X/Y in dB versus drive angular frequency |

**Do not submit only the figures or only the notebook.** No laboratory data are needed. Supplied numbers are teaching examples, not your group's measured parameters.

## Run order and exact edit
1. Read A and do its hand calculations, then run **Cell A**.
2. Run **Cell B** with `J_test = 1.0`. Predict what doubling impulse will change. Then change **only `J_test = 1.0` to `J_test = 2.0` in Cell B**, rerun Cell B, and use this two-curve comparison in your submission.
3. Run **Cell C**, then **Cell D**, with their values unchanged. Answer E in your document; it has no code cell.
4. Keep m, c, k, force amplitude, drive frequency, initial conditions, spring stiffnesses and endpoint amplitude unchanged. Do not edit functions or integration settings.
5. In Colab's left **Files** panel, refresh, open the menu beside each PNG and select **Download**. Insert the three images with readable axes, units and a one-sentence caption. Rerunning a cell overwrites its PNG; download after the final required run.

Runtime > Run all runs the default examples, but **does not complete the required J_test change or your written answers**. If you restart or rerun all, repeat the J_test = 2 run before downloading Figure 1.


## A — Predict the frequencies before running
**Given:** m = 1 kg, c = 1 N s/m, k = 100 N/m.

**Calculate on paper/in your document:**
- $\omega_n=\sqrt{k/m}$ [rad/s], $\zeta=c/(2\sqrt{mk})$ [dimensionless], and $\omega_d=\omega_n\sqrt{1-\zeta^2}$ [rad/s].
- $f_n=\omega_n/(2\pi)$ and $f_d=\omega_d/(2\pi)$ [Hz].

**Run Cell A to check. Submit:** your substitutions, results with units, and 2–3 sentences distinguishing natural frequency, damped frequency and imposed drive frequency. **No figure for A. Change nothing in Cell A.**

**Variables:** m = total moving mass; c = linear damping coefficient; k = equivalent stiffness; x = displacement [m]; t = time [s]. Dots on x mean time derivatives.


In [ ]:
# CELL A — setup and frequency check: keep all values unchanged
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
m, c, k = 1.0, 1.0, 100.0
wn = np.sqrt(k/m)
alpha = c/(2*m)
zeta = c/(2*np.sqrt(m*k))
wd = wn*np.sqrt(1-zeta**2)
print(f'omega_n = {wn:.6f} rad/s; omega_d = {wd:.6f} rad/s')
print(f'f_n = {wn/(2*np.pi):.6f} Hz; f_d = {wd/(2*np.pi):.6f} Hz; zeta = {zeta:.6f}')


## B — Double a short impulse
For an ideal instantaneous impulse J [N s] applied from rest, $m\ddot{x}+c\dot{x}+kx=0$ after contact, with $x(0)=0$ and $\dot{x}(0^+)=J/m$.

The response is $x(t)=J\exp(-\alpha t)\sin(\omega_dt)/(m\omega_d)$, where $\alpha=c/(2m)$ [s$^{-1}$].

**Do:**
1. Run Cell B once with `J_test = 1.0`.
2. Before editing, predict how displacement amplitude and oscillation frequency will change if impulse doubles.
3. Change **only `J_test = 1.0` to `J_test = 2.0`** and rerun Cell B. The reference J = 1 curve stays fixed, so you can compare both on one plot.

**Submit:** Figure 1 (`PreLab5_Impulse_Comparison.png`) from the J_test = 2 run. State your prediction, compare it with the curves and explain using the equation above. State why a hand tap in the laboratory does not provide a known numerical J without a force-time measurement. **No separate fourth figure is required.**


In [ ]:
# CELL B — the ONLY student edit is this line: 1.0 -> 2.0
J_test = 1.0  # N s; set to 2.0 for the required final comparison
assert J_test > 0
J_reference = 1.0
t = np.arange(0, 20.001, 0.01)
shape = np.exp(-alpha*t)*np.sin(wd*t)/(m*wd)
x_reference = J_reference*shape
x_test = J_test*shape
fig, ax = plt.subplots()
ax.plot(t, x_reference, label='Reference J = 1 N s')
ax.plot(t, x_test, '--', label=f'Test J = {J_test:g} N s')
ax.set(xlabel='Time after impulse (s)', ylabel='Displacement (m)')
ax.grid(alpha=0.25); ax.legend(); fig.tight_layout()
fig.savefig('PreLab5_Impulse_Comparison.png', dpi=300)
plt.show()
print('Test/reference maximum-absolute-displacement ratio:', np.max(np.abs(x_test))/np.max(np.abs(x_reference)))


## C — Separate startup motion from the sustained response
**Model:** $m\ddot{x}+c\dot{x}+kx=F_0\cos(\omega t)$.

**Given:** F0 = 1 N; drive angular frequency $\omega=4\pi$ rad/s; initial displacement x(0) = 0.1 m; initial velocity = 0. Here F0 is the peak force amplitude, not the endpoint displacement used in D.

**Do:** Run Cell C unchanged. It solves the full equation and compares it with the steady sinusoid. The transient is their difference.

**Submit:** Figure 2 (`PreLab5_Transient_Steady.png`). Calculate the drive frequency in Hz. Identify the transient and steady parts, explain why the transient decays, and state which frequency remains at late time. Refer to the plotted curves; a plot alone is not an answer.


In [ ]:
# CELL C — direct-force example: keep all values unchanged
F0, omega = 1.0, 4*np.pi
Hforce = 1/complex(k-m*omega**2, c*omega)
ss = F0*abs(Hforce)*np.cos(omega*t + np.angle(Hforce))
sol = solve_ivp(lambda t,z: [z[1], (F0*np.cos(omega*t)-c*z[1]-k*z[0])/m],
               [t[0], t[-1]], [0.1, 0.0], t_eval=t, rtol=1e-9, atol=1e-11)
assert sol.success
transient = sol.y[0]-ss
fig, ax = plt.subplots()
ax.plot(t, sol.y[0], label='Full response')
ax.plot(t, ss, '--', label='Steady response')
ax.plot(t, transient, ':', label='Transient = full - steady')
ax.set(xlabel='Time (s)', ylabel='Displacement (m)')
ax.grid(alpha=0.25); ax.legend(); fig.tight_layout()
fig.savefig('PreLab5_Transient_Steady.png', dpi=300)
plt.show()


## D — Connect the simulation to the laboratory apparatus
**Actual input:** the motor moves a spring end; it does not apply the direct force of C.

$$m\ddot{x}+c\dot{x}+(k_1+k_2)x=k_2y(t),\qquad y(t)=Y\cos(\omega t).$$

Here k1 = fixed-side spring stiffness, k2 = driven-side spring stiffness [N/m]; y = moving endpoint displacement [m]; Y = endpoint **peak** amplitude [m]; X = cart peak amplitude [m]. The damper acts against the stationary track. Use k1 = 45 N/m, k2 = 55 N/m, Y = 0.01 m and the same m, c as A. Thus k1 + k2 equals the earlier k.

**Do:** Run Cell D unchanged. It produces the displacement-gain curve and independently compares the analytical gain with a numerical ODE solution at the supplied drive frequency. It fits only the late part of that simulated record.

**Submit:**
1. Figure 3 (`PreLab5_Frequency_Response.png`); its vertical axis is $20\log_{10}(X/Y)$, not X itself.
2. Calculate the low-frequency limit X/Y = k2/(k1+k2), and explain why it is generally not 1.
3. Copy the printed numerical gain, analytical gain and relative difference into a three-entry table. Explain in 1–2 sentences what their agreement checks. This is a simulation consistency check, not validation against measured lab data.


In [ ]:
# CELL D — driven-end apparatus: keep all values unchanged
k1, k2, Y = 45.0, 55.0, 0.01
w = np.logspace(-1, 2, 400)
hy = k2/(k1+k2-m*w*w + 1j*c*w)
fig, ax = plt.subplots()
ax.semilogx(w, 20*np.log10(abs(hy)))
ax.set(xlabel='Drive angular frequency (rad/s)', ylabel='Displacement ratio X/Y (dB)')
ax.grid(alpha=0.25); fig.tight_layout()
fig.savefig('PreLab5_Frequency_Response.png', dpi=300)
plt.show()
tt = np.arange(0, 40.001, 0.01)
s = solve_ivp(lambda t,z: [z[1], (k2*Y*np.cos(omega*t)-c*z[1]-(k1+k2)*z[0])/m],
              [tt[0], tt[-1]], [0.0, 0.0], t_eval=tt, rtol=1e-9, atol=1e-11)
assert s.success
q = tt > 30
A = np.column_stack([np.cos(omega*tt[q]), np.sin(omega*tt[q]), np.ones(sum(q))])
b = np.linalg.lstsq(A, s.y[0,q], rcond=None)[0]
numeric_gain = np.hypot(b[0], b[1])/Y
theory_gain = abs(k2/complex(k1+k2-m*omega**2, c*omega))
relative_error = abs(numeric_gain/theory_gain-1)
assert relative_error < 0.002
print(f'Numerical X/Y: {numeric_gain:.9f}')
print(f'Analytical X/Y: {theory_gain:.9f}')
print(f'Relative difference (fraction): {relative_error:.3g}')


## E — Prepare your bench plan (no code)
**Submit:** A short table explaining how you will obtain each required quantity:
- Total moving mass, including installed rods and damper parts [kg].
- Both individual spring stiffnesses [N/m], using your Lab 4 calibration when the springs are the same.
- Motor-endpoint peak amplitude Y [m], obtained from its peak-to-peak travel.
- Actual drive frequency from measured rpm, not motor-supply voltage.
- Damping c [N s/m] from a decay with the same mass and installed damper configuration.

Then show these two worked conversions: **0.024 m input peak-to-peak → Y [m]**; **72 rpm → f [Hz] and omega [rad/s]**, for one drive cycle per revolution.

In 2–3 sentences explain why changing magnets needs a matching damping estimate, and give one check that a record has reached approximately steady amplitude before selecting the fit interval. Bring your spring IDs and Lab 4 calibration results to the lab.

## Before submitting
- [ ] One Word/PDF with name, section and headings A–E.
- [ ] A: worked frequency/damping calculations and distinction from drive frequency.
- [ ] B: prediction, explanation and Figure 1 with **both J = 1 and J = 2**.
- [ ] C: drive-frequency calculation, physical explanation and Figure 2.
- [ ] D: low-frequency calculation, numerical/analytical comparison table and Figure 3.
- [ ] E: measurement plan, worked conversions and settled-response/damping explanation.
- [ ] Three readable figures with captions. Define variables used in your equations.

Upload the document before your own section. Running code alone is not the submission. No measured laboratory data or additional programming are required for this pre-lab.
